# 01 - Pl@ntNet extraction: OUR scores + descriptor embeddings

Thin runner. All logic lives in `pcc/extract/plantnet_extract.py`,
`pcc/extract/forward.py`, `pcc/data/plantnet_zip.py`.

## What gets stored, and why

| split | source | stored | used for |
|---|---|---|---|
| `cal` | `images/val`, LTC's seeded 70% | logits + labels | calibration, delta_y |
| `proper_val` | `images/val`, the other 30% | logits + labels | extra holdout |
| `test` | `images/test` | logits + labels | evaluation |
| `train_quota` | `images/train`, quota per class | logits + labels + **embeddings** | descriptors phi(y) |

Embeddings only for `train_quota`: Sec 6.3 requires descriptors to come from TRAINING data, so
cal/test embeddings would waste Drive for nothing.

## The gate did NOT pass

Pl@ntNet runs under a **written exception** (reports/phase0_checkpoint_gate.md): the weights are
confirmed LTC's, but their released scores cannot be reproduced bit-exactly (median L-inf 1.2e-3,
attributed to image decode/resize implementation). Consequence, and it is why this notebook exists
in this form: **we compute our OWN scores for everything** - calibration, delta_y, evaluation and
every baseline - so no released-vs-ours seam exists. Cell 4 prints the exception loudly and it is
recorded in every shard manifest.

## Colab survival

Images live on **ephemeral /content**; only embeddings/scores go to Drive (Sec 3.2). Every split is
sharded with a checksummed manifest, so a killed session resumes from the last verified shard - and
a re-run after a kill produces output identical to one clean pass (tests/test_extract_resume.py).


## 1. Config - `# === EDIT ME ===`


In [ ]:
# === EDIT ME ===========================================================
REPO_URL   = ''
REPO_DIR   = 'foundation-cp'
DRIVE_ROOT = '/content/drive/MyDrive/pcc'
DATASET    = 'plantnet'
BACKBONE   = 'resnet50_ltc'

SPLITS     = ('cal', 'test', 'train_quota')   # add 'proper_val' if wanted
TRAIN_QUOTA = 100        # images/class for descriptors. 100 x 1081 ~ 108k of 306k images.
                         # Descriptor stability (reports/descriptor_stability_findings.md)
                         # crossed 0.90 at q=100 on CIFAR-100; the per-stratum study will
                         # tell us what the Pl@ntNet tail can actually supply.
SHARD_SIZE = 2000
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 42

DATA_ROOT  = '/content/plantnet_300K'
ZIP_PATH   = '/content/plantnet_300K.zip'
OUT_ROOT   = f'{DRIVE_ROOT}/embeddings/{DATASET}/{BACKBONE}'
# =======================================================================
print('OUT_ROOT =', OUT_ROOT, '| splits', SPLITS, '| train quota', TRAIN_QUOTA)


## 2. Mount Drive + repo + env + seed


In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
if REPO_URL and not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone',REPO_URL,REPO_DIR], check=True)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR if os.path.isabs(REPO_DIR) else '/content/'+REPO_DIR)
subprocess.run(['pip','install','-q','-r','requirements.txt'], check=False)
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH','')
from pcc.utils.seed import set_seed
from pcc.utils.device import get_device, gpu_name
from pcc.utils.io import environment_stamp
import torch
set_seed(SEED); DEVICE = get_device()
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('GPU:', gpu_name()); print('env:', environment_stamp()['packages'])


## 3. Gate check - PASS or a documented EXCEPTION (never silently neither)


In [ ]:
import json
PASS_MARKER = f'{DRIVE_ROOT}/gates/GATE_PASSED_{DATASET}_cal.json'
EXC_MARKER  = f'{DRIVE_ROOT}/gates/GATE_EXCEPTION_{DATASET}_cal.json'
if os.path.exists(PASS_MARKER):
    gate, UNDER_EXCEPTION = json.load(open(PASS_MARKER)), False
    print('gate PASSED')
elif os.path.exists(EXC_MARKER):
    gate, UNDER_EXCEPTION = json.load(open(EXC_MARKER)), True
    print('!' * 74)
    print('RUNNING UNDER A GATE EXCEPTION - the checkpoint gate FAILED.')
    print('!' * 74)
    for k, v in gate['results'].items():
        if isinstance(v, dict) and 'pass' in v: print(f'  {k:22s} pass={v["pass"]}')
    print()
    print(gate['rationale'])
else:
    raise SystemExit('BLOCKED: no gate marker. Run 00_verify_checkpoint first.')
CKPT_SHA = gate['checksums']['checkpoint']
print('ckpt sha256:', CKPT_SHA[:16], '...')


## 4. Load the gated checkpoint and verify its checksum still matches


In [ ]:
import glob
from pcc.data.ltc_datasets import NUM_CLASSES
from pcc.extract.backbones import load_ltc_resnet50
from pcc.eval.score_repro import sha256_file

cands = [f for f in glob.glob(f'{DRIVE_ROOT}/checkpoints/ltc_models/**/best-{DATASET}-model.pth',
                             recursive=True) if 'focal' not in f.lower()]
assert cands, 'checkpoint not found on Drive'
CKPT_PATH = cands[0]
sha = sha256_file(CKPT_PATH)
assert sha == CKPT_SHA, f'checkpoint CHANGED since the gate ran: {sha[:16]} != {CKPT_SHA[:16]}'
model = load_ltc_resnet50(CKPT_PATH, NUM_CLASSES[DATASET], DEVICE)
print('loaded', CKPT_PATH, '| head', model.fc.out_features, '| sha verified')


## 5. Make sure the needed images exist (selective unzip, idempotent)

`cal` and `proper_val` both come from the **val** directory; `test` from the **test** directory;
`train_quota` extracts only a QUOTA per class rather than all ~28 GB of train.

**The archive is downloaded here if missing.** `/content` is ephemeral, so after a session
restart the 32 GB zip is gone; this cell resumes/re-downloads it (MD5-verified) rather than
assuming notebook 00 left it behind.

**The archive layout is READ, not assumed.** A previous version shelled out to
`unzip 'plantnet_300K/images/test/*'` and died with `exit status 9` (nothing matched) even though
`images/val` extracted fine - the archive does not necessarily use the split names one expects.
This cell prints the real split names first, and extraction now goes through Python `zipfile`
(idempotent, completes interrupted runs, and reports the actual layout on a miss).


In [ ]:
import shutil
from pcc.data.plantnet_zip import available_splits, ensure_split
from pcc.data.plantnet_download import ensure_zip

print('free /content: %.1f GB' % (shutil.disk_usage('/content').free/1e9))

# /content is EPHEMERAL: after a session restart the 32 GB archive is gone. This
# notebook therefore downloads/resumes it itself instead of assuming notebook 00
# left it behind (which is what broke on 2026-08-05).
print(ensure_zip(ZIP_PATH))

avail = available_splits(ZIP_PATH)
print('splits present in the archive:', avail)

# our split names -> archive directory names
SRC_DIR = {'cal': 'val', 'proper_val': 'val', 'test': 'test', 'train_quota': 'train'}
needed = {SRC_DIR[sp] for sp in SPLITS}
missing = [d for d in needed if d not in avail]
if missing:
    raise SystemExit(f'archive has no split(s) {missing}. Present: {sorted(avail)}. '
                     f'Adjust SRC_DIR / SPLITS to the real names above.')

for d in sorted(needed):
    if d == 'train':
        print(ensure_split(ZIP_PATH, '/content', 'train', quota=TRAIN_QUOTA, seed=SEED))
    else:
        have = (len([e for e in os.scandir(f'{DATA_ROOT}/images/{d}') if e.is_dir()])
                if os.path.isdir(f'{DATA_ROOT}/images/{d}') else 0)
        if have >= NUM_CLASSES[DATASET]:
            print(f'  {d}: {have} class folders already present'); continue
        print(ensure_split(ZIP_PATH, '/content', d))
print('free /content: %.1f GB' % (shutil.disk_usage('/content').free/1e9))


## 6. Extract each split (sharded, checksummed, resumable)

Safe to re-run: a killed session resumes from the last verified shard, and a completed split is
a no-op. Provenance drift (a different checkpoint) is refused rather than mixed in.


In [ ]:
from pcc.extract.plantnet_extract import extract_split

results = {}
for sp in SPLITS:
    results[sp] = extract_split(sp, data_root=DATA_ROOT, out_root=OUT_ROOT,
                                model=model, device=DEVICE, ckpt_sha=CKPT_SHA,
                                quota=TRAIN_QUOTA if sp == 'train_quota' else None,
                                seed=SEED, shard_size=SHARD_SIZE,
                                batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                                under_gate_exception=UNDER_EXCEPTION)
    print(' ->', results[sp])


## 7. Verify every manifest, then write the report


In [ ]:
import time
from pcc.data.manifest import verify_manifest
from pcc.utils.io import write_report

ok = True
for sp, r in results.items():
    v = verify_manifest(r['out_dir'])
    complete = v['n_verified_samples'] >= r['n_expected']
    ok = ok and v['ok'] and complete
    print(f"{sp:12s} verified={v['ok']} samples={v['n_verified_samples']}/{r['n_expected']}"
          f" complete={complete}")
    if v['missing'] or v['corrupt']:
        print('   missing:', v['missing'][:3], 'corrupt:', v['corrupt'][:3])

report = write_report('pcc/reports', f'01_extract_{DATASET}',
    hypothesis='logits/labels (all splits) and penultimate embeddings (train quota) extracted '
               'from the gated LTC checkpoint, sharded with verified checksums',
    pass_criteria='every split complete (n_verified >= n_expected) and every shard checksum '
                  'verifies; provenance records the gate-exception status; embeddings stored '
                  'only for train_quota (Sec 6.3 descriptors come from training data)',
    config=dict(dataset=DATASET, backbone=BACKBONE, splits=list(SPLITS),
                train_quota=TRAIN_QUOTA, shard_size=SHARD_SIZE,
                ckpt_sha256=CKPT_SHA, under_gate_exception=UNDER_EXCEPTION,
                scores_source='OURS, not LTC released'),
    seed=SEED, results={k: {kk: vv for kk, vv in v.items()} for k, v in results.items()},
    conclusion=('PASS - all splits extracted and verified' if ok else
                'INCOMPLETE - re-run this notebook, extraction resumes'),
    started_at=time.time())
print('report:', report)
print()
print('KEEP the zip until you are sure no further split is needed; then os.remove(ZIP_PATH).')
if UNDER_EXCEPTION:
    print('REMINDER: these scores are OURS. Every Pl@ntNet result must say so and cite')
    print('reports/phase0_checkpoint_gate.md.')
